# Free boson one point functions

This funciton evaluates the torus one point functions

$$
\left\langle:\!\partial X\partial X\!:\right\rangle,
\qquad
\left\langle:\!\partial X\bar\partial X\!:\right\rangle
$$

directly from a discretized ribbon graph. It then compares them with the exact flat torus expressions. The comparison is independent of the Weyl anomaly in the partition function.

In [2]:
from string_amplitudes import (
    compute_period_map,
    free_boson_one_point_functions,
    generate_ribbon_graphs,
    genus_one_disk_frame_one_point_functions,
    genus_one_period_matrix_one_point_functions,
)

## Period-matrix expressions

In the flat coordinate $u\sim u+1\sim u+\tau$, the exact expressions are

$$
\left\langle:\!\partial_uX\bar\partial_{\bar u}X\!:\right\rangle
=\frac{1}{2\tau_2},
\qquad
\left\langle:\!\partial_uX\partial_uX\!:\right\rangle
=\frac{\theta_1'''(0\mid\tau)}{6\theta_1'(0\mid\tau)}
+\frac{\pi}{2\tau_2}.
$$

The ribbon graph calculation uses a disc coordinate $z$. If $u(z)=\int^z\omega$ for a normalized holomorphic one form $\omega$ is the transformation between two coordinate systems, the mixed operator transforms as a primary, while the holomorphic operator also receives the Schwarzian term:

$$
\left\langle:\!\partial_zX\partial_zX\!:\right\rangle
=u'(z)^2\left\langle:\!\partial_uX\partial_uX\!:\right\rangle
-\frac{1}{12}\{u,z\}.
$$

In [4]:
# generate a ribbon graph
graph = generate_ribbon_graphs(genus=1, n_faces=1)[0]
edge_lengths = (48, 68, 100)
# get the period matrix associated with a given ribbon graph structure.
period_data = compute_period_map(
    graph,
    edge_lengths,
    period_quadrature_order=128,
)

# compute the one point functions in the disk frame
numerical_disk = free_boson_one_point_functions(graph, edge_lengths)
# compute the exact flat torus one points
exact_flat = genus_one_period_matrix_one_point_functions(
    period_data.period_matrix
)
#compute the exact flat torus one point functions, and transform exactly to the disk frame.
exact_disk = genus_one_disk_frame_one_point_functions(period_data)

print("Period matrix:", period_data.period_matrix)
print("Exact flat-torus values:", exact_flat)
print("Numerical disc values:  ", numerical_disk)
print("Exact disc values:      ", exact_disk)

Period matrix: [[0.58021404+0.74337055j]]
Exact flat-torus values: FreeBosonOnePointFunctions(holomorphic=(0.14991557840873826-0.1698963840344848j), mixed=(0.6726120645790611+0j))
Numerical disc values:   FreeBosonOnePointFunctions(holomorphic=(-0.0022001322486815593-0.013507373545267773j), mixed=(0.15240199084581146+4.336808689942018e-19j))
Exact disc values:       FreeBosonOnePointFunctions(holomorphic=(-0.0020146573157070313-0.01348338265041138j), mixed=(0.15244361044666008+0j))


## Increase the number of discretization points

With fixed edge lengths, as the total number of points used in the discretization is increased, the accuracy of the computation should increase.

In [5]:
base_lengths = (12, 17, 25)
# basically just repeat the same calculation as above, just increasing the total
# number of points.
print("boundary length   holomorphic relative difference   mixed relative difference")
for scale in (1, 2, 4, 8):
    lengths = tuple(scale * length for length in base_lengths)
    data = compute_period_map(
        graph,
        lengths,
        period_quadrature_order=128,
    )
    numerical = free_boson_one_point_functions(graph, lengths)
    exact = genus_one_disk_frame_one_point_functions(data)
    holomorphic_difference = abs(numerical.holomorphic / exact.holomorphic - 1.0)
    mixed_difference = abs(numerical.mixed / exact.mixed - 1.0)
    print(
        f"{2 * sum(lengths):15d}"
        f"   {holomorphic_difference:31.6e}"
        f"   {mixed_difference:25.6e}"
    )

boundary length   holomorphic relative difference   mixed relative difference
            108                      5.597320e-02                1.656565e-03
            216                      2.749436e-02                6.774932e-04
            432                      1.371813e-02                2.730164e-04
            864                      6.890561e-03                1.092823e-04
